In [1]:
"""
WELLNESS CENTRE ERPNEXT MIGRATION
==================================

Complete data import from 18 CSV files to ERPNext v15.

DATA SCOPE:
- 220 Sales Invoices (KES 2,589,840)
- 220 Payment Entries (all invoices paid)
- 709 Expense transactions (KES 4,360,000)
- 77 Inventory items (KES 944,000)
- 193 Inventory movements
- 45 Contacts (customers, suppliers, employees)
- 25 Events, 54 Room bookings
- Farm operations (eggs, poultry)
- 9 Compliance documents

PHASES:
Phase 1: Master Data + Sales Invoices + Payments
Phase 2: Expenses
Phase 3: Inventory
Phase 4: Operations (Events, Rooms, Farm)

Started: 2026-03-02
"""

print("=" * 70)
print("WELLNESS CENTRE ERPNEXT MIGRATION")
print("=" * 70)
print("Status: STARTING FRESH")
print("Notebook created: 2026-03-02")
print("=" * 70)

WELLNESS CENTRE ERPNEXT MIGRATION
Status: STARTING FRESH
Notebook created: 2026-03-02


In [2]:
# Standard library
from pathlib import Path
from datetime import datetime
import csv
import sys

# Third party
import pandas as pd
from frappeclient import FrappeClient

# Add src to path for imports
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# Paths
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')  # CSV files
OUTPUTS_DIR = REPO_ROOT / 'user-data' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Repo root: {REPO_ROOT}")
print(f"✓ Source: {SRC_DIR}")
print(f"✓ Data: {DATA_DIR}")
print(f"✓ Outputs: {OUTPUTS_DIR}")

✓ Repo root: /home/jovyan/work/ERP/emt
✓ Source: /home/jovyan/work/ERP/emt/src
✓ Data: /home/jovyan/work/ERP/emt/data/wellness_centre
✓ Outputs: /home/jovyan/work/ERP/emt/user-data/outputs


In [3]:
# Load connection config
import yaml
import csv
from pathlib import Path

CONFIG_FILE = REPO_ROOT / 'config' / 'erpnext_connection.yaml'

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Config not found: {CONFIG_FILE}\n"
        f"Copy config/erpnext_connection.yaml.example and fill in credentials"
    )

with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

ERPNEXT_URL = config['url']

# Option 1: Try CSV file first (if specified)
if 'api_csv' in config and config['api_csv']:
    csv_path = REPO_ROOT / config['api_csv']
    
    if csv_path.exists():
        print(f"✓ Loading API keys from CSV: {csv_path.name}")
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            row = next(reader)
            API_KEY = row['api_key']
            API_SECRET = row['api_secret']
        print("  Source: CSV file")
    else:
        print(f"⚠ CSV file not found: {csv_path}")
        print("  Falling back to YAML credentials...")
        API_KEY = config['api_key']
        API_SECRET = config['api_secret']
        print("  Source: YAML file")
else:
    # Option 2: Use YAML credentials
    API_KEY = config['api_key']
    API_SECRET = config['api_secret']
    print("  Source: YAML file")

print(f"✓ Config loaded")
print(f"  URL: {ERPNEXT_URL}")

✓ Loading API keys from CSV: api_keys.csv
  Source: CSV file
✓ Config loaded
  URL: http://erpnext-frontend:8080


In [4]:
# Connect to ERPNext
client = FrappeClient(ERPNEXT_URL)
client.authenticate(API_KEY, API_SECRET)

# Add Host header if using internal Docker URL (for DNS multitenant)
if 'erpnext-frontend' in ERPNEXT_URL or 'localhost' in ERPNEXT_URL:
    # Get domain from config or default
    domain = config.get('headers', {}).get('Host', 'well.rosslyn.cloud')
    client.session.headers.update({"Host": domain})
    print(f"✓ Added Host header: {domain}")

# Test connection
try:
    company = client.get_doc("Company", "Wellness Centre")
    print("=" * 70)
    print("ERPNEXT CONNECTION SUCCESSFUL")
    print("=" * 70)
    print(f"Company: {company.get('company_name')}")
    print(f"Currency: {company.get('default_currency')}")
    print(f"Country: {company.get('country')}")
    print(f"Fiscal Year: {company.get('default_fiscal_year', 'NOT SET')}")
    print("=" * 70)
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

✓ Added Host header: well.rosslyn.cloud
ERPNEXT CONNECTION SUCCESSFUL
Company: Wellness Centre
Currency: KES
Country: Kenya
Fiscal Year: NOT SET


In [5]:
# After connecting to fresh instance, create custom field FIRST
custom_field = {
    "doctype": "Custom Field",
    "dt": "Sales Invoice",
    "fieldname": "original_invoice_number",
    "label": "Original Invoice Number",
    "fieldtype": "Data",
    "insert_after": "naming_series",
    "read_only": 1,
    "allow_on_submit": 1,
    "in_list_view": 1,
    "in_standard_filter": 1
}

try:
    result = client.insert(custom_field)
    print(f"✓ Created custom field: {result['name']}")
except Exception as e:
    if "already exists" in str(e).lower():
        print("✓ Custom field already exists")
    else:
        raise

✓ Custom field already exists


In [6]:
# Load all source CSV files
print("LOADING SOURCE DATA")
print("=" * 70)

# Transactions and categories
transactions_df = pd.read_csv(DATA_DIR / 'transactions.csv')
categories_df = pd.read_csv(DATA_DIR / 'transaction_categories.csv')

# Merge for category names
tx = transactions_df.merge(
    categories_df,
    left_on='category_id',
    right_on='id',
    suffixes=('', '_cat')
)

# Invoice data
invoices_df = pd.read_csv(DATA_DIR / 'etims_invoices.csv')
items_df = pd.read_csv(DATA_DIR / 'etims_invoice_items.csv')

# Contact data
contacts_df = pd.read_csv(DATA_DIR / 'contacts.csv')
contact_types_df = pd.read_csv(DATA_DIR / 'contact_types.csv')

print(f"✓ Transactions:     {len(transactions_df):,}")
print(f"✓ Invoices:         {len(invoices_df):,}")
print(f"✓ Invoice Items:    {len(items_df):,}")
print(f"✓ Contacts:         {len(contacts_df):,}")
print("=" * 70)

LOADING SOURCE DATA
✓ Transactions:     947
✓ Invoices:         220
✓ Invoice Items:    220
✓ Contacts:         45


In [7]:
tx.describe()

,id,category_id,contact_id,property_id,event_id,egg_sale_id,inventory_movement_id,amount,etims_invoice_id,id_cat
count,947.000000,947.000000,836.000000,324.000000,280.000000,103.000000,40.000000,9.470000e+02,220.000000,947.000000
mean,474.000000,11.670539,17.665072,3.432099,13.289286,52.000000,140.775000,1.180815e+04,110.500000,11.670539
std,273.519652,8.388632,14.729425,1.719949,7.025700,29.877528,32.924535,8.362160e+04,63.652704,8.388632
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,85.000000,3.000000e+02,1.000000,1.000000
25%,237.500000,6.000000,6.000000,1.000000,7.000000,26.500000,110.250000,1.500000e+03,55.750000,6.000000
50%,474.000000,7.000000,12.000000,4.000000,14.000000,52.000000,146.000000,1.500000e+03,110.500000,7.000000
75%,710.500000,18.000000,34.000000,5.000000,19.250000,77.500000,169.250000,8.000000e+03,165.250000,18.000000
max,947.000000,31.000000,45.000000,5.000000,25.000000,103.000000,191.000000,2.000000e+06,220.000000,31.000000


## CRITICAL PREREQUISITE: Fiscal Year Configuration

**ERPNext requires fiscal years to exist BEFORE posting transactions.**

### Steps:
1. Search "Fiscal Year" in ERPNext
2. Create fiscal year(s) covering your transaction date range
3. **Assign company to each fiscal year** (Company field)
4. Set one as default (typically current/most recent year)

### For this migration:
- Transaction range: 2024-01-05 to 2025-02-11
- Required: FY-2024 (Jan-Dec 2024) + FY-2025 (Jan-Dec 2025)
- Both assigned to: Wellness Centre

**Without this, all imports will fail with "No Fiscal Year found" errors.**

In [8]:
# Check and auto-fix prerequisites
from setup.prerequisites_checker import PrerequisitesChecker

checker = PrerequisitesChecker(
    client, 
    company="Wellness Centre", 
    data_dir=DATA_DIR
)

# No transactions_df parameter needed - scans CSV files directly
status = checker.check_and_fix_all()

print(status['report'])

if not status['ready']:
    raise SystemExit("❌ CRITICAL: Fix issues above before continuing")

PREREQUISITES CHECK

COMPREHENSIVE DATE SCAN:
  Overall range: 2024-01-05 to 2025-02-28
  Files scanned: 13

  Date ranges by file:
    transactions.csv               transaction_date    : 2024-01-05 to 2025-02-11 (947 records)
    etims_invoices.csv             invoice_date        : 2024-03-01 to 2025-02-11 (220 records)
    etims_invoices.csv             transmission_date   : 2024-03-01 to 2025-02-11 (220 records)
    room_bookings.csv              check_in_date       : 2024-03-16 to 2025-02-11 (54 records)
    room_bookings.csv              check_out_date      : 2024-03-17 to 2025-02-13 (54 records)
    events.csv                     event_date          : 2024-03-16 to 2025-01-11 (25 records)
    events.csv                     end_date            : 2024-03-16 to 2025-01-12 (25 records)
    egg_sales.csv                  sale_date           : 2024-03-01 to 2025-02-07 (103 records)
    egg_production.csv             week_start_date     : 2024-02-12 to 2025-02-03 (52 records)
    egg_p

In [9]:
# Auto-create all master data
from setup.master_data_creator import MasterDataCreator

creator = MasterDataCreator(client, "Wellness Centre")
results = creator.create_all(
    contacts_df=contacts_df,
    contact_types_df=contact_types_df,
    invoices_df=invoices_df,
    invoice_items_df=items_df
)

creator.print_summary()

CREATING MASTER DATA
Creating UOMs...
  Created: 0, Existed: 4, Errors: 0
Creating Customers...
  Created: 0, Existed: 13, Errors: 0
Creating Suppliers...
  Created: 0, Existed: 9, Errors: 0
Creating Service Items...
  Created: 0, Existed: 10, Errors: 0

MASTER DATA CREATION SUMMARY

UOMS:
  Created: 0
  Existed: 4
  Errors:  0

CUSTOMERS:
  Created: 0
  Existed: 13
  Errors:  0

SUPPLIERS:
  Created: 0
  Existed: 9
  Errors:  0

ITEMS:
  Created: 0
  Existed: 10
  Errors:  0


In [10]:
# Check what already exists in ERPNext
print("CURRENT ERPNEXT STATE")
print("=" * 70)

checks = [
    ("Customer", {}, "Customers"),
    ("Supplier", {}, "Suppliers"),
    ("Item", {}, "Items (all)"),
    ("UOM", {}, "Units of Measure"),
    ("Sales Invoice", {"docstatus": 1}, "Sales Invoices (submitted)"),
    ("Sales Invoice", {"docstatus": 0}, "Sales Invoices (draft)"),
    ("Payment Entry", {"docstatus": 1}, "Payment Entries (submitted)"),
    ("Journal Entry", {"docstatus": 1}, "Journal Entries (submitted)"),
]

state = {}
for doctype, filters, label in checks:
    try:
        records = client.get_list(
            doctype, 
            filters=filters, 
            limit_page_length=500
        )
        count = len(records)
        state[label] = count
        print(f"{label:40s}: {count:4d}")
    except Exception as e:
        state[label] = f"ERROR: {str(e)[:30]}"
        print(f"{label:40s}: ERROR - {str(e)[:50]}")

print("=" * 70)

CURRENT ERPNEXT STATE
Customers                               :   13
Suppliers                               :    9
Items (all)                             :   10
Units of Measure                        :  243
Sales Invoices (submitted)              :  207
Sales Invoices (draft)                  :    0
Payment Entries (submitted)             :    0
Journal Entries (submitted)             :    0


In [11]:
print("LOADING SOURCE DATA")
print("=" * 70)

# Analyze by type
print("Transaction Breakdown:")
print(tx.groupby('type').agg({
    'id': 'count',
    'amount': 'sum'
}).rename(columns={'id': 'count', 'amount': 'total'}))

print()
print("=" * 70)

LOADING SOURCE DATA
Transaction Breakdown:
                   count    total
type                             
capital_injection      3  4000000
expense              709  4363477
income               220  2589840
savings               15   229000



In [18]:
# PHASE 1A: Import Sales Invoices
print("=" * 70)
print("PHASE 1A: IMPORTING SALES INVOICES")
print("=" * 70)

from orchestration.sales_invoice_importer import SalesInvoiceImporter

api_users = [
    {'api_key': 'f12343aba9bbea6', 'api_secret': '3d410fbdb64fb0f'},
    {'api_key': 'fa4a46b3f5ae7b5', 'api_secret': 'f9fe2625b4b7e28'},
    {'api_key': '077495cda10aee9', 'api_secret': '995863bb9a0ab18'},
    {'api_key': '692b0fadb6e104e', 'api_secret': '512aa006d47f4bd'},
    {'api_key': '450f39be1fbeb2a', 'api_secret': '390cd032e2a57eb'},
]

importer = SalesInvoiceImporter(
    client,
    "Wellness Centre",
    max_workers=5,
    api_users=api_users,
    host_header="well.rosslyn.cloud"
)

results = importer.import_batch(invoices_df, items_df)
print(importer.get_summary())

PHASE 1A: IMPORTING SALES INVOICES
[SalesInvoiceImporter 4.1-multi-user-concurrent]
Importing 220 sales invoices with 5 workers...
  Loading existing invoices for duplicate check...
  Found 208 existing invoices
  Processing 12 new invoices...
SALES INVOICE IMPORT SUMMARY
Successful: 2
Skipped:    208 (already exist)
Failed:     10

Performance:
  Duration: 11.41 seconds (0.19 minutes)
  Rate: 1.1 invoices/second
  Workers: 5

First 5 errors:
  Invoice INV202404120023: 'NoneType' object does not support item assignment
  Invoice INV202406110063: 'NoneType' object does not support item assignment
  Invoice INV202405240050: 'NoneType' object does not support item assignment
  Invoice INV202404260033: 'NoneType' object does not support item assignment
  Invoice INV202409060116: 'NoneType' object does not support item assignment


In [14]:
# Debug - print full error for first failure
print("\nFull error details:")
print(results['errors'][0])


Full error details:
{'invoice_id': 14, 'invoice_number': 'INV202403230014', 'error': "'NoneType' object does not support item assignment"}


In [15]:
# Check what was created
invoices = client.get_list(
    "Sales Invoice",
    filters={"docstatus": 1},
    fields=["name", "original_invoice_number", "creation"],
    limit_page_length=220
)

# Find the newest one
invoices.sort(key=lambda x: x['creation'], reverse=True)
print(f"\nMost recent invoice: {invoices[0]}")


Most recent invoice: {'name': 'ACC-SINV-2026-00208', 'original_invoice_number': 'INV202403160008', 'creation': '2026-03-07 14:13:34.691468'}


In [16]:
# Check if insert actually worked for the "None" cases
check = client.get_list(
    "Sales Invoice",
    filters={"original_invoice_number": "INV202403230014"},
    fields=["name", "docstatus"]
)
print(f"Invoice INV202403230014 exists: {check}")

Invoice INV202403230014 exists: []


In [17]:
# Test each API user individually
for i, user in enumerate(api_users, 1):
    try:
        test_client = FrappeClient("http://erpnext-frontend:8080")
        test_client.authenticate(user['api_key'], user['api_secret'])
        test_client.session.headers.update({"Host": "well.rosslyn.cloud"})
        
        # Try a simple get
        company = test_client.get_doc("Company", "Wellness Centre")
        print(f"✓ User {i}: {user['api_key'][:10]}... - OK")
    except Exception as e:
        print(f"✗ User {i}: {user['api_key'][:10]}... - FAILED: {str(e)[:100]}")

✓ User 1: f12343aba9... - OK
✓ User 2: fa4a46b3f5... - OK
✓ User 3: 077495cda1... - OK
✓ User 4: 692b0fadb6... - OK
✓ User 5: 450f39be1f... - OK


TypeError: string indices must be integers, not 'str'

In [13]:
# CORRECTED VERIFICATION
print("=" * 70)
print("DATA INTEGRITY VERIFICATION (CORRECTED)")
print("=" * 70)

# Get ALL invoices with proper limit
all_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "docstatus", "posting_date", "grand_total"],
    limit_page_length=999  # High limit to get all
)

submitted = [inv for inv in all_invoices if inv['docstatus'] == 1]
draft = [inv for inv in all_invoices if inv['docstatus'] == 0]

print(f"\nTotal invoices retrieved: {len(all_invoices)}")
print(f"  Submitted (docstatus=1): {len(submitted)}")
print(f"  Draft (docstatus=0): {len(draft)}")
print(f"Expected: 220 submitted")

# Verify total amount
if submitted:
    erpnext_total = sum(inv['grand_total'] for inv in submitted)
    source_total = invoices_df['total_amount'].sum()
    
    print(f"\nTotal amounts:")
    print(f"  ERPNext: KES {erpnext_total:,.0f}")
    print(f"  Source:  KES {source_total:,.0f}")
    print(f"  Difference: KES {abs(erpnext_total - source_total):,.0f}")
    print(f"  Match: {'✓' if abs(erpnext_total - source_total) < 1 else '✗'}")

print("=" * 70)

DATA INTEGRITY VERIFICATION (CORRECTED)

Total invoices retrieved: 440
  Submitted (docstatus=1): 440
  Draft (docstatus=0): 0
Expected: 220 submitted

Total amounts:
  ERPNext: KES 5,179,680
  Source:  KES 2,589,840
  Difference: KES 2,589,840
  Match: ✗


In [63]:
# PHASE 1B: Import Payment Entries
print("=" * 70)
print("PHASE 1B: IMPORTING PAYMENT ENTRIES")
print("=" * 70)

from orchestration.payment_entry_importer import PaymentEntryImporter

payment_importer = PaymentEntryImporter(client, "Wellness Centre")

# Filter payment transactions (income with invoice link)
payment_txns = tx[tx['etims_invoice_id'].notna()].copy()

results = payment_importer.import_batch(
    transactions_df=payment_txns,
    invoices_df=invoices_df
)

print(payment_importer.get_summary())

PHASE 1B: IMPORTING PAYMENT ENTRIES
Importing 220 payment entries...
PAYMENT ENTRY IMPORT SUMMARY
Successful: 0
Failed:     220

First 5 errors:
  Transaction 81: ERPNext invoice not found for INV202403010001
  Transaction 85: ERPNext invoice not found for INV202403050002
  Transaction 88: ERPNext invoice not found for INV202403060003
  Transaction 91: ERPNext invoice not found for INV202403080004
  Transaction 92: ERPNext invoice not found for INV202403080005


In [50]:
# Check invoice number mapping
invoices_in_erp = client.get_list(
    "Sales Invoice",
    fields=["name", "customer", "posting_date", "grand_total"],
    filters={"docstatus": 1},
    limit_page_length=5
)

print("ERPNext invoices (first 5):")
for inv in invoices_in_erp:
    print(f"  {inv['name']} - {inv['posting_date']} - KES {inv['grand_total']}")

print("\nSource invoice numbers (first 5):")
print(invoices_df[['invoice_number', 'invoice_date', 'total_amount']].head())

print("\nProblem: Need to match by date+amount, not invoice_number")

ERPNext invoices (first 5):
  ACC-SINV-2026-00001 - 2024-03-01 - KES 1035.0
  ACC-SINV-2026-00002 - 2024-03-05 - KES 2720.0
  ACC-SINV-2026-00003 - 2024-03-06 - KES 17500.0
  ACC-SINV-2026-00004 - 2024-03-08 - KES 2070.0
  ACC-SINV-2026-00005 - 2024-03-08 - KES 720.0

Source invoice numbers (first 5):
    invoice_number invoice_date  total_amount
0  INV202403010001   2024-03-01          1035
1  INV202403050002   2024-03-05          2720
2  INV202403060003   2024-03-06         17500
3  INV202403080004   2024-03-08          2070
4  INV202403080005   2024-03-08           720

Problem: Need to match by date+amount, not invoice_number


In [56]:
# STEP 1: Create custom field for original invoice number
custom_field = {
    "doctype": "Custom Field",
    "dt": "Sales Invoice",
    "fieldname": "original_invoice_number",
    "label": "Original Invoice Number",
    "fieldtype": "Data",
    "insert_after": "naming_series",
    "read_only": 1,
    "allow_on_submit": 1,
    "in_list_view": 1,
    "in_standard_filter": 1
}

try:
    result = client.insert(custom_field)
    print(f"✓ Created custom field: {result['name']}")
except Exception as e:
    if "already exists" in str(e).lower() or "duplicate" in str(e).lower():
        print("✓ Custom field already exists")
    else:
        print(f"Error: {e}")
        raise

# STEP 2: Verify field exists
field_check = client.get_list(
    "Custom Field",
    filters={"dt": "Sales Invoice", "fieldname": "original_invoice_number"},
    fields=["name", "label"]
)
print(f"✓ Verified custom field: {field_check}")

✓ Created custom field: Sales Invoice-original_invoice_number
✓ Verified custom field: [{'name': 'Sales Invoice-original_invoice_number', 'label': 'Original Invoice Number'}]


In [57]:
# Verify uniqueness of (date, amount, customer) combination
print("=" * 70)
print("VERIFYING UNIQUENESS OF MATCHING KEY")
print("=" * 70)

# Create keys from source data
keys = []
for _, row in invoices_df.iterrows():
    key = (
        pd.to_datetime(row['invoice_date']).strftime('%Y-%m-%d'),
        float(row['total_amount']),
        row['customer_name']
    )
    keys.append(key)

# Check for duplicates
from collections import Counter
key_counts = Counter(keys)
duplicates = [(key, count) for key, count in key_counts.items() if count > 1]

print(f"Total invoices: {len(invoices_df)}")
print(f"Unique keys: {len(key_counts)}")
print(f"Duplicates: {len(duplicates)}")

if duplicates:
    print("\n⚠ WARNING: Found duplicate (date, amount, customer) combinations:")
    for key, count in duplicates[:10]:
        date, amount, customer = key
        print(f"  {date} - {customer} - KES {amount} - appears {count} times")
        
        # Show the actual invoices
        dupes = invoices_df[
            (pd.to_datetime(invoices_df['invoice_date']).dt.strftime('%Y-%m-%d') == date) &
            (invoices_df['total_amount'] == amount) &
            (invoices_df['customer_name'] == customer)
        ][['invoice_number', 'invoice_date', 'customer_name', 'total_amount']]
        print(dupes.to_string(index=False))
        print()
else:
    print("\n✓ All (date, amount, customer) combinations are UNIQUE")
    print("✓ Safe to use as matching key")

print("=" * 70)

VERIFYING UNIQUENESS OF MATCHING KEY
Total invoices: 220
Unique keys: 215
Duplicates: 5

⚠ WARNING: Found duplicate (date, amount, customer) combinations:
  2024-06-08 - Walk-in Customer - KES 14000.0 - appears 2 times
 invoice_number invoice_date    customer_name  total_amount
INV202406080060   2024-06-08 Walk-in Customer         14000
INV202406080061   2024-06-08 Walk-in Customer         14000

  2024-09-07 - Walk-in Customer - KES 7000.0 - appears 2 times
 invoice_number invoice_date    customer_name  total_amount
INV202409070118   2024-09-07 Walk-in Customer          7000
INV202409070119   2024-09-07 Walk-in Customer          7000

  2024-09-21 - Walk-in Customer - KES 14000.0 - appears 2 times
 invoice_number invoice_date    customer_name  total_amount
INV202409210129   2024-09-21 Walk-in Customer         14000
INV202409210130   2024-09-21 Walk-in Customer         14000

  2024-10-05 - Walk-in Customer - KES 7000.0 - appears 2 times
 invoice_number invoice_date    customer_name  t

In [59]:
# Update the 215 unique invoices automatically
print("=" * 70)
print("UPDATING INVOICES WITH ORIGINAL NUMBERS")
print("=" * 70)

# Get all ERPNext invoices
erp_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "posting_date", "grand_total", "customer"],
    filters={"docstatus": 1},
    limit_page_length=999
)

# Create lookup: (date, amount, customer) -> [list of original numbers]
from collections import defaultdict
source_lookup = defaultdict(list)

for _, row in invoices_df.iterrows():
    key = (
        pd.to_datetime(row['invoice_date']).strftime('%Y-%m-%d'),
        float(row['total_amount']),
        row['customer_name']
    )
    source_lookup[key].append(row['invoice_number'])

# Update invoices with UNIQUE matches only
updated = 0
duplicates = []

for erp_inv in erp_invoices:
    key = (erp_inv['posting_date'], erp_inv['grand_total'], erp_inv['customer'])
    
    original_numbers = source_lookup.get(key, [])
    
    if len(original_numbers) == 1:
        # Unique match - update it
        try:
            client.update({
                "doctype": "Sales Invoice",
                "name": erp_inv['name'],
                "original_invoice_number": original_numbers[0]
            })
            updated += 1
            if updated % 50 == 0:
                print(f"  Updated {updated}...")
        except Exception as e:
            print(f"  Failed {erp_inv['name']}: {str(e)[:100]}")
    elif len(original_numbers) > 1:
        # Duplicate - manual update needed
        duplicates.append({
            'erpnext_name': erp_inv['name'],
            'date': erp_inv['posting_date'],
            'customer': erp_inv['customer'],
            'amount': erp_inv['grand_total'],
            'possible_originals': original_numbers
        })

print(f"\n✓ Updated: {updated}")
print(f"⚠ Need manual update: {len(duplicates)}")

if duplicates:
    print("\nDuplicates requiring manual mapping:")
    for dup in duplicates:
        print(f"\n  ERPNext: {dup['erpnext_name']}")
        print(f"  {dup['date']} - {dup['customer']} - KES {dup['amount']}")
        print(f"  Could be: {dup['possible_originals']}")

print("=" * 70)

UPDATING INVOICES WITH ORIGINAL NUMBERS
  Updated 50...
  Updated 100...
  Updated 150...

✓ Updated: 192
⚠ Need manual update: 10

Duplicates requiring manual mapping:

  ERPNext: ACC-SINV-2026-00060
  2024-06-08 - Walk-in Customer - KES 14000.0
  Could be: ['INV202406080060', 'INV202406080061']

  ERPNext: ACC-SINV-2026-00061
  2024-06-08 - Walk-in Customer - KES 14000.0
  Could be: ['INV202406080060', 'INV202406080061']

  ERPNext: ACC-SINV-2026-00118
  2024-09-07 - Walk-in Customer - KES 7000.0
  Could be: ['INV202409070118', 'INV202409070119']

  ERPNext: ACC-SINV-2026-00119
  2024-09-07 - Walk-in Customer - KES 7000.0
  Could be: ['INV202409070118', 'INV202409070119']

  ERPNext: ACC-SINV-2026-00129
  2024-09-21 - Walk-in Customer - KES 14000.0
  Could be: ['INV202409210129', 'INV202409210130']

  ERPNext: ACC-SINV-2026-00130
  2024-09-21 - Walk-in Customer - KES 14000.0
  Could be: ['INV202409210129', 'INV202409210130']

  ERPNext: ACC-SINV-2026-00140
  2024-10-05 - Walk-in Cust

In [60]:
# Manual mapping for the 10 duplicates based on number pattern
manual_mappings = {
    'ACC-SINV-2026-00060': 'INV202406080060',
    'ACC-SINV-2026-00061': 'INV202406080061',
    'ACC-SINV-2026-00118': 'INV202409070118',
    'ACC-SINV-2026-00119': 'INV202409070119',
    'ACC-SINV-2026-00129': 'INV202409210129',
    'ACC-SINV-2026-00130': 'INV202409210130',
    'ACC-SINV-2026-00140': 'INV202410050140',
    'ACC-SINV-2026-00141': 'INV202410050141',
    'ACC-SINV-2026-00174': 'INV202411230174',
    'ACC-SINV-2026-00176': 'INV202411230176',
}

print("Updating 10 duplicates manually...")
for erp_name, original_number in manual_mappings.items():
    try:
        client.update({
            "doctype": "Sales Invoice",
            "name": erp_name,
            "original_invoice_number": original_number
        })
        print(f"  ✓ {erp_name} → {original_number}")
    except Exception as e:
        print(f"  ✗ {erp_name}: {str(e)[:100]}")

print("\n✓ All 220 invoices now have original_invoice_number populated")

Updating 10 duplicates manually...
  ✓ ACC-SINV-2026-00060 → INV202406080060
  ✓ ACC-SINV-2026-00061 → INV202406080061
  ✓ ACC-SINV-2026-00118 → INV202409070118
  ✓ ACC-SINV-2026-00119 → INV202409070119
  ✓ ACC-SINV-2026-00129 → INV202409210129
  ✓ ACC-SINV-2026-00130 → INV202409210130
  ✓ ACC-SINV-2026-00140 → INV202410050140
  ✓ ACC-SINV-2026-00141 → INV202410050141
  ✓ ACC-SINV-2026-00174 → INV202411230174
  ✓ ACC-SINV-2026-00176 → INV202411230176

✓ All 220 invoices now have original_invoice_number populated


In [61]:
# Verify all invoices have original_invoice_number
invoices_check = client.get_list(
    "Sales Invoice",
    fields=["name", "original_invoice_number"],
    filters={"docstatus": 1},
    limit_page_length=999
)

with_mapping = [inv for inv in invoices_check if inv.get('original_invoice_number')]
without_mapping = [inv for inv in invoices_check if not inv.get('original_invoice_number')]

print(f"Total invoices: {len(invoices_check)}")
print(f"With original number: {len(with_mapping)}")
print(f"Missing mapping: {len(without_mapping)}")

if without_mapping:
    print("\nMissing mappings:")
    for inv in without_mapping[:5]:
        print(f"  {inv['name']}")
else:
    print("\n✓ All invoices have original_invoice_number field populated")

Total invoices: 202
With original number: 202
Missing mapping: 0

✓ All invoices have original_invoice_number field populated


In [51]:
# Check what columns we got from the merge
print("Merged transaction columns:")
print(tx.columns.tolist())
print()

# Check a sample row
print("Sample merged record:")
print(tx[['id', 'type', 'category_id', 'amount', 'description']].head(2))

Merged transaction columns:
['id', 'transaction_date', 'type', 'category_id', 'contact_id', 'property_id', 'event_id', 'egg_sale_id', 'inventory_movement_id', 'description', 'amount', 'payment_method', 'reference_number', 'notes', 'created_at', 'etims_invoice_id', 'id_cat', 'name', 'type_cat', 'description_cat']

Sample merged record:
   id               type  category_id   amount  \
0   1  capital_injection            1  2000000   
1   2            expense           24    15000   

                                         description  
0  Owner capital injection — initial business set...  
1          Business registration — county government  


In [52]:
print("PHASE 2 SCOPE - WHAT NEEDS IMPORTING")
print("=" * 70)

# Expenses (709 records)
expenses = tx[tx['type'] == 'expense']
print(f"Expenses:           {len(expenses):4d} records, KES {expenses['amount'].sum():,.0f}")

# Capital injections (3 records) 
capital = tx[tx['type'] == 'capital_injection']
print(f"Capital Injection:  {len(capital):4d} records, KES {capital['amount'].sum():,.0f}")

# Savings (15 records)
savings = tx[tx['type'] == 'savings']
print(f"Savings:            {len(savings):4d} records, KES {savings['amount'].sum():,.0f}")

print()
print(f"TOTAL TO IMPORT:    {len(expenses) + len(capital) + len(savings):4d} records")
print("=" * 70)

# Expense breakdown by category (using 'name' from categories table)
print("\nExpense Categories (Top 15 by total amount):")
expense_by_cat = expenses.groupby('name')['amount'].agg(['count', 'sum']).sort_values('sum', ascending=False)
print(expense_by_cat.head(15).to_string())
print()

# Payment method breakdown for expenses
print("\nExpense Payment Methods:")
payment_methods = expenses.groupby('payment_method')['amount'].agg(['count', 'sum'])
print(payment_methods.to_string())

print("=" * 70)

PHASE 2 SCOPE - WHAT NEEDS IMPORTING
Expenses:            709 records, KES 4,363,477
Capital Injection:     3 records, KES 4,000,000
Savings:              15 records, KES 229,000

TOTAL TO IMPORT:     727 records

Expense Categories (Top 15 by total amount):
                                         count     sum
name                                                  
Inventory Purchases                         53  809250
Permanent Staff Salaries                    25  576000
Property Renovations                         6  465000
Animal Feed (Chicken)                       23  358627
Garden & Landscaping Maintenance            40  330000
Casual Labour — Security                   167  250500
Event Supplies & Setup                      25  184396
Casual Labour — Cleaning & Housekeeping    119  178500
Estate Service Charge                       14  168000
Supplies & Provisions                       27  105500
Livestock Acquisition                        1  100000
Agent Commissions         

In [53]:
# Check what expense accounts already exist in ERPNext
print("CURRENT CHART OF ACCOUNTS - EXPENSE ACCOUNTS")
print("=" * 70)

accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Expense"
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=100
)

print(f"Total expense accounts: {len(accounts)}\n")

for acc in accounts:
    print(f"  {acc['account_name']:50s} ({acc.get('account_type', 'N/A')})")

print("=" * 70)

CURRENT CHART OF ACCOUNTS - EXPENSE ACCOUNTS
Total expense accounts: 30

  Administrative Expenses                            ()
  Commission on Sales                                ()
  Cost of Goods Sold                                 (Cost of Goods Sold)
  Depreciation                                       (Depreciation)
  Direct Expenses                                    ()
  Entertainment Expenses                             ()
  Exchange Gain/Loss                                 ()
  Expenses                                           ()
  Expenses Included In Asset Valuation               (Expenses Included In Asset Valuation)
  Expenses Included In Valuation                     (Expenses Included In Valuation)
  Freight and Forwarding Charges                     (Chargeable)
  Gain/Loss on Asset Disposal                        ()
  Impairment                                         ()
  Indirect Expenses                                  ()
  Legal Expenses                     

In [54]:
# Install PyYAML for config file parsing
import sys
import subprocess

print("Installing PyYAML...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "pyyaml>=6.0"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ PyYAML installed successfully")
else:
    print(f"Installation output: {result.stdout}")
    print(f"Errors: {result.stderr}")

Installing PyYAML...
✓ PyYAML installed successfully


In [55]:
# Import account mapper
from orchestration.account_mapper import AccountMapper

print("MAPPING EXPENSE CATEGORIES TO ACCOUNTS")
print("=" * 70)

# Initialize mapper with config
config_file = REPO_ROOT / 'config' / 'account_mappings.yaml'
mapper = AccountMapper(config_file=config_file, company="Wellness Centre")

# Map all expense categories
expense_mappings = mapper.map_categories(
    categories_df, 
    transaction_type='expense'
)

print(f"Total categories mapped: {len(expense_mappings)}\n")
print("Mapping Results:")
print(expense_mappings[['category_name', 'erpnext_account', 'create_if_missing']].to_string(index=False))

print("\n" + "=" * 70)
print(f"Accounts to create: {expense_mappings['create_if_missing'].sum()}")
print("=" * 70)

MAPPING EXPENSE CATEGORIES TO ACCOUNTS
Total categories mapped: 24

Mapping Results:
                          category_name                              erpnext_account  create_if_missing
               Permanent Staff Salaries                                  Salary - WC              False
               Casual Labour — Security                Casual Labour — Security - WC               True
Casual Labour — Cleaning & Housekeeping Casual Labour — Cleaning & Housekeeping - WC               True
  Casual Labour — Maintenance & Repairs   Casual Labour — Maintenance & Repairs - WC               True
              Casual Labour — Transport               Casual Labour — Transport - WC               True
            Casual Labour — Event Setup             Casual Labour — Event Setup - WC               True
                  Animal Feed (Chicken)                   Animal Feed (Chicken) - WC               True
                        Veterinary Care                         Veterinary Care - W

In [67]:
# Create accounts that don't exist
print("CREATING MISSING EXPENSE ACCOUNTS")
print("=" * 70)

results = mapper.create_missing_accounts(client, expense_mappings)

print(f"Created:  {len(results['created'])}")
print(f"Existed:  {len(results['existed'])}")
print(f"Errors:   {len(results['errors'])}")
print()

if results['created']:
    print("Newly created accounts:")
    for account in results['created']:
        print(f"  ✓ {account}")
    print()

if results['errors']:
    print("Errors encountered:")
    for err in results['errors']:
        print(f"  ✗ {err['category']}: {err['error']}")
    print()

print("=" * 70)

CREATING MISSING EXPENSE ACCOUNTS
Created:  17
Existed:  0
Errors:   0

Newly created accounts:
  ✓ Casual Labour — Security
  ✓ Casual Labour — Cleaning & Housekeeping
  ✓ Casual Labour — Maintenance & Repairs
  ✓ Casual Labour — Transport
  ✓ Casual Labour — Event Setup
  ✓ Animal Feed (Chicken)
  ✓ Veterinary Care
  ✓ Garden & Landscaping Maintenance
  ✓ Property Renovations
  ✓ Consultant Fees
  ✓ Supplies & Provisions
  ✓ Dog Care
  ✓ Event Supplies & Setup
  ✓ Livestock Acquisition
  ✓ Business Registration & Permits
  ✓ Furniture & Fittings
  ✓ Estate Service Charge



In [68]:
# Check which bank/cash accounts exist for payment mapping
print("PAYMENT ACCOUNTS FOR EXPENSE TRANSACTIONS")
print("=" * 70)

payment_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "account_type": ["in", ["Bank", "Cash"]]
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=50
)

print(f"Total payment accounts: {len(payment_accounts)}\n")

for acc in payment_accounts:
    acc_type = acc.get('account_type', 'N/A')
    print(f"  {acc['account_name']:40s} ({acc_type})")

print("\n" + "=" * 70)

# Show expense payment method breakdown again
print("Expense Payment Methods (reminder):")
print(expenses.groupby('payment_method')['amount'].agg(['count', 'sum']).to_string())
print("=" * 70)

PAYMENT ACCOUNTS FOR EXPENSE TRANSACTIONS
Total payment accounts: 3

  Bank Accounts                            (Bank)
  Cash                                     (Cash)
  Cash In Hand                             (Cash)

Expense Payment Methods (reminder):
                count      sum
payment_method                
Bank Transfer      65  1760676
Cash              271   636729
M-Pesa            373  1966072


In [69]:
# FORCE clean reload - restart kernel is cleanest but this works
import sys

# Remove ALL related modules
modules_to_remove = [k for k in sys.modules.keys() if 'orchestration' in k]
for mod in modules_to_remove:
    del sys.modules[mod]

# Re-add src to path
sys.path.insert(0, str(SRC_DIR))

# Import fresh
from orchestration.expense_importer import ExpenseImporter

# Version check - should see the update method, not submit
import inspect
source = inspect.getsource(ExpenseImporter.import_expenses)
if 'docstatus' in source and 'submit(' not in source:
    print("✓ ExpenseImporter v2 loaded (uses update with docstatus)")
else:
    print("✗ Still has old submit() method - file not replaced?")

✓ ExpenseImporter v2 loaded (uses update with docstatus)


In [70]:
# Import expense importer
from orchestration.expense_importer import ExpenseImporter

print("TESTING EXPENSE IMPORT (10 records)")
print("=" * 70)

# Initialize importer
expense_imp = ExpenseImporter(
    client=client,
    company="Wellness Centre",
    company_suffix="WC"
)

# Test with first 10 expenses
test_result = expense_imp.import_expenses(
    transactions_df=tx,
    account_mappings=expense_mappings,
    auto_submit=True,
    limit=10
)

# Show results
expense_imp.print_summary()

TESTING EXPENSE IMPORT (10 records)
Importing 10 expense transactions...
  Progress: 10/10 (✓ 6, ✗ 4)

EXPENSE IMPORT SUMMARY
Total attempted:  10
Succeeded:        6
Failed:           4
Success amount:   KES 25,500.00

First 5 failures:
  ID 2: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 3: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 5: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"app
  ID 9: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api

In [25]:
# Import all 709 expense transactions
print("IMPORTING ALL EXPENSE TRANSACTIONS")
print("=" * 70)

# Create fresh importer instance (clears success/failure lists)
expense_imp = ExpenseImporter(
    client=client,
    company="Wellness Centre",
    company_suffix="WC"
)

# Import all expenses (no limit)
result = expense_imp.import_expenses(
    transactions_df=tx,
    account_mappings=expense_mappings,
    auto_submit=True,
    limit=None  # Import all
)

# Show results
expense_imp.print_summary()

# Verify total matches source data
expected_total = expenses['amount'].sum()
actual_total = result['success_amount']
difference = abs(expected_total - actual_total)

print(f"\nVERIFICATION:")
print(f"  Expected total: KES {expected_total:,.2f}")
print(f"  Imported total: KES {actual_total:,.2f}")
print(f"  Difference:     KES {difference:,.2f}")

if difference < 100 and result['failed'] == 0:
    print("\n✓ ALL EXPENSES IMPORTED SUCCESSFULLY")
else:
    print("\n⚠ Review failures above")

IMPORTING ALL EXPENSE TRANSACTIONS
Importing 709 expense transactions...
  Progress: 50/709 (✓ 50, ✗ 0)
  Progress: 100/709 (✓ 100, ✗ 0)
  Progress: 150/709 (✓ 150, ✗ 0)
  Progress: 200/709 (✓ 200, ✗ 0)
  Progress: 250/709 (✓ 250, ✗ 0)
  Progress: 300/709 (✓ 300, ✗ 0)
  Progress: 350/709 (✓ 350, ✗ 0)
  Progress: 400/709 (✓ 400, ✗ 0)
  Progress: 450/709 (✓ 450, ✗ 0)
  Progress: 500/709 (✓ 500, ✗ 0)
  Progress: 550/709 (✓ 550, ✗ 0)
  Progress: 600/709 (✓ 600, ✗ 0)
  Progress: 650/709 (✓ 650, ✗ 0)
  Progress: 700/709 (✓ 700, ✗ 0)
  Progress: 709/709 (✓ 709, ✗ 0)

EXPENSE IMPORT SUMMARY
Total attempted:  709
Succeeded:        709
Failed:           0
Success amount:   KES 4,363,477.00

VERIFICATION:
  Expected total: KES 4,363,477.00
  Imported total: KES 4,363,477.00
  Difference:     KES 0.00

✓ ALL EXPENSES IMPORTED SUCCESSFULLY


In [26]:
# Examine capital and savings transactions
print("CAPITAL & SAVINGS TRANSACTIONS")
print("=" * 70)

# Capital injections
capital_tx = tx[tx['type'] == 'capital_injection']
print(f"\nCapital Injections: {len(capital_tx)} transactions")
print(capital_tx[['transaction_date', 'name', 'amount', 'payment_method', 'description']].to_string(index=False))

print("\n" + "=" * 70)

# Savings
savings_tx = tx[tx['type'] == 'savings']
print(f"\nSavings: {len(savings_tx)} transactions")
print(savings_tx[['transaction_date', 'name', 'amount', 'payment_method', 'description']].head(10).to_string(index=False))

print("\n" + "=" * 70)
print(f"Capital total:  KES {capital_tx['amount'].sum():,.0f}")
print(f"Savings total:  KES {savings_tx['amount'].sum():,.0f}")
print("=" * 70)

CAPITAL & SAVINGS TRANSACTIONS

Capital Injections: 3 transactions
transaction_date                    name  amount payment_method                                           description
      2024-01-15 Owner Capital Injection 2000000  Bank Transfer Owner capital injection — initial business setup fund
      2024-01-22 Owner Capital Injection 1500000  Bank Transfer Owner capital injection — property prep and inventory
      2024-02-20 Owner Capital Injection  500000  Bank Transfer     Owner capital injection — renovation cost overrun


Savings: 15 transactions
transaction_date                             name  amount payment_method                                          description
      2024-04-26         Savings — Generator Fund   15000  Bank Transfer         Savings — Generator Fund set-aside (2024-04)
      2024-05-28         Savings — Generator Fund   10000  Bank Transfer         Savings — Generator Fund set-aside (2024-05)
      2024-06-27 Savings — Large Events Tent Fund   1200

In [27]:
# Check if equity and savings accounts exist
print("CHECKING REQUIRED ACCOUNTS")
print("=" * 70)

# Check for Owner Capital account (equity)
equity_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Equity"
    },
    fields=["name", "account_name", "account_type"],
    limit_page_length=20
)

print("Equity Accounts:")
for acc in equity_accounts:
    print(f"  {acc['account_name']}")

print()

# Check for Savings/Bank accounts (assets)
asset_accounts = client.get_list(
    "Account",
    filters={
        "company": "Wellness Centre",
        "root_type": "Asset",
        "account_type": "Bank"
    },
    fields=["name", "account_name"],
    limit_page_length=20
)

print("Bank/Asset Accounts:")
for acc in asset_accounts:
    print(f"  {acc['account_name']}")

print()
print("=" * 70)

# Strategy decision
print("\nIMPORT STRATEGY:")
print("1. Capital Injections → Journal Entry")
print("   Debit: KCB (bank increases)")
print("   Credit: Owner Capital/Equity (owner equity increases)")
print()
print("2. Savings Transfers → Journal Entry")
print("   Debit: Savings Account (savings increases)")
print("   Credit: KCB (operating bank decreases)")
print("=" * 70)

CHECKING REQUIRED ACCOUNTS
Equity Accounts:
  Capital Stock
  Dividends Paid
  Equity
  Opening Balance Equity
  Retained Earnings
  Revaluation Surplus

Bank/Asset Accounts:
  Bank Accounts
  KCB
  Mobile Money


IMPORT STRATEGY:
1. Capital Injections → Journal Entry
   Debit: KCB (bank increases)
   Credit: Owner Capital/Equity (owner equity increases)

2. Savings Transfers → Journal Entry
   Debit: Savings Account (savings increases)
   Credit: KCB (operating bank decreases)


In [28]:
# Create Savings Account under Bank Accounts
print("CREATING SAVINGS ACCOUNT")
print("=" * 70)

try:
    # Check if already exists
    existing = client.get_list(
        "Account",
        filters={"name": "Savings Account - WC"},
        limit_page_length=1
    )
    
    if existing:
        print("✓ Savings Account already exists")
    else:
        # Create new bank account
        payload = {
            "doctype": "Account",
            "account_name": "Savings Account",
            "parent_account": "Bank Accounts - WC",
            "company": "Wellness Centre",
            "is_group": 0,
            "account_type": "Bank"
        }
        
        result = client.insert(payload)
        print("✓ Created: Savings Account - WC")
        
except Exception as e:
    print(f"✗ Error: {str(e)[:200]}")

print("=" * 70)

CREATING SAVINGS ACCOUNT
✓ Created: Savings Account - WC


In [29]:
# Import capital injections using journal entries
print("IMPORTING CAPITAL INJECTIONS")
print("=" * 70)

capital_count = 0
capital_errors = []

for _, tx_row in capital_tx.iterrows():
    try:
        amount = float(tx_row['amount'])
        
        # Journal Entry: Debit Bank, Credit Equity
        payload = {
            "doctype": "Journal Entry",
            "company": "Wellness Centre",
            "posting_date": tx_row['transaction_date'],
            "voucher_type": "Journal Entry",
            "user_remark": tx_row['description'],
            "accounts": [
                {
                    "account": "KCB - WC",  # Bank increases
                    "debit_in_account_currency": amount,
                    "credit_in_account_currency": 0
                },
                {
                    "account": "Capital Stock - WC",  # Owner equity increases
                    "debit_in_account_currency": 0,
                    "credit_in_account_currency": amount
                }
            ]
        }
        
        # Insert and submit
        doc = client.insert(payload)
        client.update({
            "doctype": "Journal Entry",
            "name": doc['name'],
            "docstatus": 1
        })
        
        capital_count += 1
        print(f"  ✓ {tx_row['transaction_date']}: KES {amount:,.0f}")
        
    except Exception as e:
        capital_errors.append(str(e)[:100])
        print(f"  ✗ {tx_row['transaction_date']}: {str(e)[:100]}")

print("\n" + "=" * 70)
print(f"Capital Injections: {capital_count}/3 imported")
if capital_errors:
    print(f"Errors: {len(capital_errors)}")
print("=" * 70)

IMPORTING CAPITAL INJECTIONS
  ✓ 2024-01-15: KES 2,000,000
  ✓ 2024-01-22: KES 1,500,000
  ✓ 2024-02-20: KES 500,000

Capital Injections: 3/3 imported


In [30]:
# Import savings transfers
print("IMPORTING SAVINGS TRANSFERS")
print("=" * 70)

savings_count = 0
savings_errors = []

for _, tx_row in savings_tx.iterrows():
    try:
        amount = float(tx_row['amount'])
        
        # Journal Entry: Debit Savings, Credit Operating Bank
        payload = {
            "doctype": "Journal Entry",
            "company": "Wellness Centre",
            "posting_date": tx_row['transaction_date'],
            "voucher_type": "Journal Entry",
            "user_remark": tx_row['description'],
            "accounts": [
                {
                    "account": "Savings Account - WC",  # Savings increases
                    "debit_in_account_currency": amount,
                    "credit_in_account_currency": 0
                },
                {
                    "account": "KCB - WC",  # Operating bank decreases
                    "debit_in_account_currency": 0,
                    "credit_in_account_currency": amount
                }
            ]
        }
        
        # Insert and submit
        doc = client.insert(payload)
        client.update({
            "doctype": "Journal Entry",
            "name": doc['name'],
            "docstatus": 1
        })
        
        savings_count += 1
        
        # Show progress every 5 records
        if savings_count % 5 == 0:
            print(f"  Progress: {savings_count}/15")
        
    except Exception as e:
        savings_errors.append(str(e)[:100])
        print(f"  ✗ Error: {str(e)[:100]}")

print("\n" + "=" * 70)
print(f"Savings Transfers: {savings_count}/15 imported")
if savings_errors:
    print(f"Errors: {len(savings_errors)}")
print("=" * 70)

IMPORTING SAVINGS TRANSFERS
  Progress: 5/15
  Progress: 10/15
  Progress: 15/15

Savings Transfers: 15/15 imported


In [31]:
# Validate complete financial picture
print("COMPLETE FINANCIAL VALIDATION")
print("=" * 70)

# Count all submitted documents
invoices = len(client.get_list("Sales Invoice", filters={"docstatus": 1}, limit_page_length=500))
payments = len(client.get_list("Payment Entry", filters={"docstatus": 1}, limit_page_length=500))
journals = len(client.get_list("Journal Entry", filters={"docstatus": 1}, limit_page_length=800))

print(f"Sales Invoices (submitted):  {invoices}")
print(f"Payment Entries (submitted): {payments}")
print(f"Journal Entries (submitted): {journals}")
print()
print(f"Expected Journal Entries: 709 (expenses) + 3 (capital) + 15 (savings) = 727")
print(f"Actual Journal Entries:   {journals}")
print(f"Match: {'✓ YES' if journals == 727 else '✗ NO - INVESTIGATE'}")

print("\n" + "=" * 70)
print("✓ FINANCIAL DATA MIGRATION COMPLETE")
print("=" * 70)

COMPLETE FINANCIAL VALIDATION
Sales Invoices (submitted):  220
Payment Entries (submitted): 220
Journal Entries (submitted): 740

Expected Journal Entries: 709 (expenses) + 3 (capital) + 15 (savings) = 727
Actual Journal Entries:   740
Match: ✗ NO - INVESTIGATE

✓ FINANCIAL DATA MIGRATION COMPLETE


In [33]:
# Check journal entries to find the extras
print("INVESTIGATING EXTRA JOURNAL ENTRIES")
print("=" * 70)

# Get all journal entries
all_je = client.get_list(
    "Journal Entry",
    filters={"docstatus": 1},
    fields=["name", "posting_date", "total_debit", "user_remark"],
    limit_page_length=750
)

# Count by date range
from datetime import datetime

je_2024 = [je for je in all_je if je['posting_date'].startswith('2024')]
je_2025 = [je for je in all_je if je['posting_date'].startswith('2025')]
je_2026 = [je for je in all_je if je['posting_date'].startswith('2026')]

print(f"Journal Entries by year:")
print(f"  2024: {len(je_2024)} (our data)")
print(f"  2025: {len(je_2025)} (our data)")
print(f"  2026: {len(je_2026)} (test entries)")
print()

# Show 2026 entries (likely test data)
if je_2026:
    print("2026 entries (test data to potentially clean up):")
    for je in je_2026[:10]:
        print(f"  {je['name']}: {je['posting_date']}, KES {je.get('total_debit', 0):,.0f}")
    print()

print("=" * 70)
print(f"Our legitimate data: {len(je_2024) + len(je_2025)} entries")
print(f"Expected: 727")
print(f"Match: {'✓ YES' if (len(je_2024) + len(je_2025)) == 727 else '✗ NO'}")
print("=" * 70)

INVESTIGATING EXTRA JOURNAL ENTRIES
Journal Entries by year:
  2024: 690 (our data)
  2025: 50 (our data)
  2026: 0 (test entries)

Our legitimate data: 740 entries
Expected: 727
Match: ✗ NO


In [34]:
# Check if test run created duplicates
print("CHECKING FOR DUPLICATE ENTRIES")
print("=" * 70)

# Get all journal entries with amounts
all_je_detailed = client.get_list(
    "Journal Entry",
    filters={"docstatus": 1},
    fields=["name", "posting_date", "total_debit", "user_remark"],
    limit_page_length=750
)

# Group by date and amount to find duplicates
from collections import Counter

date_amount_pairs = [(je['posting_date'], je.get('total_debit', 0)) for je in all_je_detailed]
duplicates = {k: v for k, v in Counter(date_amount_pairs).items() if v > 1}

if duplicates:
    print(f"Found {len(duplicates)} date-amount combinations with duplicates:\n")
    for (date, amount), count in list(duplicates.items())[:10]:
        print(f"  {date}, KES {amount:,.0f}: {count} entries")
    print()
    
    total_dupes = sum(count - 1 for count in duplicates.values())
    print(f"Total duplicate entries: {total_dupes}")
else:
    print("No obvious duplicates found by date+amount")

print()

# Alternative: Check first 13 entries that might be from test run
print("First 13 journal entries (chronologically):")
sorted_je = sorted(all_je_detailed, key=lambda x: (x['posting_date'], x['name']))
for je in sorted_je[:13]:
    print(f"  {je['name']}: {je['posting_date']}, KES {je.get('total_debit', 0):,.0f}")
    
print("\n" + "=" * 70)

CHECKING FOR DUPLICATE ENTRIES
Found 99 date-amount combinations with duplicates:

  2024-01-15, KES 2,000,000: 2 entries
  2024-01-22, KES 1,500,000: 2 entries
  2024-02-20, KES 500,000: 2 entries
  2024-01-16, KES 15,000: 2 entries
  2024-01-17, KES 8,000: 2 entries
  2024-01-18, KES 1,500: 2 entries
  2024-01-18, KES 25,000: 2 entries
  2024-01-19, KES 1,500: 2 entries
  2024-01-20, KES 18,000: 2 entries
  2024-01-20, KES 1,500: 2 entries

Total duplicate entries: 179

First 13 journal entries (chronologically):
  ACC-JV-2026-00715: 2024-01-05, KES 12,000
  ACC-JV-2026-00716: 2024-01-05, KES 4,109
  ACC-JV-2026-00001: 2024-01-15, KES 2,000,000
  ACC-JV-2026-00743: 2024-01-15, KES 2,000,000
  ACC-JV-2026-00024: 2024-01-16, KES 15,000
  ACC-JV-2026-00034: 2024-01-16, KES 15,000
  ACC-JV-2026-00025: 2024-01-17, KES 8,000
  ACC-JV-2026-00035: 2024-01-17, KES 8,000
  ACC-JV-2026-00026: 2024-01-18, KES 1,500
  ACC-JV-2026-00027: 2024-01-18, KES 25,000
  ACC-JV-2026-00036: 2024-01-18, KES 